# 07 — Gate de instabilidade no OD da estação EF01 (CETESB)

**Objetivo:** bater a régua do OD (sazonal-naive 0,1525 rolante / 0,1550 holdout) sem prever o dia — prevendo **se amanhã muda**. Um gate barato escolhe por origem: regime calmo → sazonal-naive; instável → desafiante (`lstnet_ft` do 06, variante com `lgbm_A`). Mesmo janelamento do 00b–06 (L=8640/H=288, split 70/15/15 + holdout de 10 dias, segmento limpo 01/06 → 21/07).
**Braços:** (i) **gate-rule** — limiar único no hindcast trailing (1 parâmetro); (ii) **gate-lr** — logística em 5 features causais; (iii) **oracle** — melhor escolha ex-post por origem (teto do método, não é modelo).
**Custo:** nenhum treino pesado — só reuso dos checkpoints do 06 (`lstnet_ft_od.pt` + `lgbm_mult_A_steps.pkl`, este com fallback para modo só-`ft`).
**Dados:** `dados/ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.

In [ ]:
import json
import pickle
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LogisticRegression
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "07-gate-od"
FT_CKPT = ROOT / "resultados" / "06-refit-od" / "modelos" / "lstnet_ft_od.pt"
LGBM_A_PKL = ROOT / "resultados" / "06-refit-od" / "modelos" / "lgbm_mult_A_steps.pkl"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00b) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
SEG_FIM = "2026-07-21 01:05"
HOLDOUT_DIAS = 10
CTX = 2016
HIST_J = 7       # hindcast trailing: últimos 7 dias-alvo
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| ft_ckpt:", FT_CKPT.exists(), "| lgbm_A:", LGBM_A_PKL.exists())

## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [ ]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
colvar = [c for c in df.columns if c != "Data hora"][0]
df = df.rename(columns={"Data hora": "ds", colvar: "y"}).sort_values("ds").reset_index(drop=True)
print(colvar, "|", df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

## 2. EDA — perfil, o gap de 16 dias e ciclo diário

In [ ]:
isna = df["y"].isna().to_numpy()
bounds = np.where(np.diff(np.concatenate([[False], isna, [False]])))[0]
runs = sorted([(bounds[i], bounds[i+1]-1) for i in range(0, len(bounds), 2)],
              key=lambda r: r[1]-r[0], reverse=True)
print("top 5 gaps:")
for a, b in runs[:5]:
    print(f"  {df.ds[a]} → {df.ds[b]}  ({(b-a+1)*5/60:.1f} h)")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvspan(pd.Timestamp("2026-07-21 01:10"), pd.Timestamp("2026-08-06 11:30"),
              color="r", alpha=0.2, label="sensor morto (16,4 dias)")
ax[0].set_title("OD EF01 — série completa (faixa vermelha = gap, fora do experimento)")
ax[0].set_ylabel("OD (mg/L)")
ax[0].legend(fontsize=8)
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do OD")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("OD por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

## 3. Limpeza + recorte do segmento limpo
Grade de 5 min, interpolação máx. 2 h e **corte em 21/07 01:05** (antes do gap). Tudo a jusante usa só o segmento 01/06 → 21/07.

In [ ]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_full = df.set_index("ds")["y"].reindex(idx)
s = s_full.loc[:SEG_FIM].interpolate(method="time", limit=INTERP_LIMIT)
print(f"segmento: {s.index.min()} → {s.index.max()} ({len(s)} slots = {len(s)*5/60/24:.1f} dias)")
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_full[amostra].index, s_full[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00b (últimos 4032 pontos do treino, período 288).

In [ ]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

## 5. Janelamento + dias-alvo alinhados
Janelas `(L=8640 → H=288)` idênticas ao 00b (split 70/15/15 sem shuffle + holdout + 10 origens diárias). Dias-alvo alinhados no mesmo horário das origens diárias (01:05): `T(d)` = 288 slots terminando em `d 01:05`; o hindcast `hind(d) = MAE(T(d−1), T(d))` é causal e dispensa modelo.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
for k, idx in {"train": tr, "val": va, "test": te, "holdout": ho}.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
E01 = ends[daily_idx[0]].time()
TE_DATE0 = ends[te[0]].date()
print("horário-alvo:", E01, "| te começa em:", TE_DATE0)

# T(d): vetor-alvo do dia d (288 slots até d 01:05), só dias com histórico suficiente
pos0105 = np.where((s.index.time == E01))[0]
Tdays, Tvec = [], {}
for p in pos0105:
    if p - H + 1 < 0:
        continue
    d = s.index[p].date()
    Tdays.append(d); Tvec[d] = s.to_numpy()[p - H + 1:p + 1]
Tdays = sorted(Tdays)
hind = {d: float(np.abs(Tvec[d] - Tvec[Tdays[Tdays.index(d) - 1]]).mean())
        for d in Tdays[1:]}
print(f"dias-alvo: {len(Tdays)} ({Tdays[0]} → {Tdays[-1]}); hind calculado p/ {len(hind)} dias")
print("hind 12→21/07:", [round(hind[d], 3) for d in Tdays if str(d) >= '2026-07-12'])

## 6. Baselines + desafiantes recarregados
Sazonal-naive/persistência/MM (iguais ao 00b) + `lstnet_ft` do 06 (dependência dura) + `lgbm_A` do 06 (opcional, com fallback para modo só-`ft` em clone fresco).

In [ ]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

ft_ckpt = torch.load(FT_CKPT, map_location="cpu", weights_only=False)
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), 2016, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - 2016 + 1
Wln2 = sliding_window_view(s.to_numpy().astype(np.float32), 2016)

@torch.no_grad()
def prevê_lstnet(idxs, state, batch=128):
    net = LSTNet1D().to(DEVICE)
    net.load_state_dict(state); net.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln2[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(net(xb, tb).numpy())
    del net
    return np.concatenate(outs)

ft_state = ft_ckpt["state"]
try:
    with open(LGBM_A_PKL, "rb") as f:
        models_A = pickle.load(f)
    HAVE_LGBM = True
    print(f"lgbm_A recarregado ({len(models_A)} modelos)")
except FileNotFoundError:
    models_A, HAVE_LGBM = None, False
    print("lgbm_A ausente — modo só-ft (rode o 06 para o braço lgbm)")

if HAVE_LGBM:
    def base_feats(Xb, E):
        cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
        phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
        cols += [phase.mean(1), phase.std(1)]
        for w in [12, 36, 144, 288]:
            cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
        cols += [Xb[:, -2016:].mean(1)]
        F = np.stack(cols, axis=1)
        em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
        return F.astype(np.float32), em, phase
    def amp_feats(Xb, phase):
        Ctx = Xb[:, -CTX:]
        Ref = Xb[:, L - SEASON - CTX:L - SEASON]
        return np.stack([Ctx.std(axis=1), Ctx.max(axis=1) - Ctx.min(axis=1),
                         phase.max(axis=1) - phase.min(axis=1),
                         Ctx.std(axis=1) / np.maximum(Ref.std(axis=1), 1e-6)], axis=1).astype(np.float32)
    def hour_sincos(em, j):
        hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
        return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)
    def prevê_lgbm(idxs):
        ii = np.asarray(idxs)
        Xb = X[ii]
        F, em, Ph = base_feats(Xb, ends[ii])
        F = np.column_stack([F, amp_feats(Xb, Ph)])
        S = snaive(Xb)
        P = np.empty((len(ii), H), dtype=np.float32)
        for j, m in enumerate(models_A):
            sh, ch = hour_sincos(em, j)
            P[:, j] = S[:, j] * np.clip(m.predict(np.column_stack([F, sh, ch])), 0.5, 1.5)
        return P

Xte, Yte = X[te], Y[te]
pred_te = cheap_preds(Xte)
print("teste rolante (baratos):")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())

## 7. Features do gate (só passado) + labels pré-`te`
Por dia-alvo `d`: `hind_mean7/max/last` (hindcast trailing), `ptp_slope7` (tendência da amplitude diária) e `ctx_std` — tudo causal. Labels: `ft` venceu o sazonal em `T(d)`? Dias de ajuste: `d < te` (limpos); `te` valida qualitativamente; holdout decide.

In [ ]:
dptp = s.to_frame("y").resample("1D").agg(ptp=("y", lambda a: float(a.max() - a.min())))["ptp"]

def feats_dia(d):
    j = Tdays.index(d)
    hh = np.array([hind[Tdays[t]] for t in range(j - HIST_J, j)])
    cal = pd.Timestamp(d) - pd.Timedelta(days=1)
    hist = dptp.loc[:cal].iloc[-HIST_J:].to_numpy()
    slope = float(np.polyfit(np.arange(len(hist)), hist, 1)[0]) if len(hist) >= 3 else 0.0
    p = int(np.where(s.index == pd.Timestamp(d) + pd.Timedelta(hours=1, minutes=5))[0][0])
    ctx = s.to_numpy()[p - CTX:p]
    return np.array([hh.mean(), hh.max(), hh[-1], slope, ctx.std()], dtype=np.float64)

FEAT_NAMES = ["hind_mean7", "hind_max7", "hind_last", "ptp_slope7", "ctx_std"]

# labels: ft × sazonal em cada dia-alvo pré-te (com ft inferido no contexto 2016)
gate_days = [d for d in Tdays if d >= pd.Timestamp("2026-06-16").date() and d < TE_DATE0]
print(f"dias p/ ajuste do gate: {len(gate_days)} ({gate_days[0]} → {gate_days[-1]})")
Fgate = np.stack([feats_dia(d) for d in gate_days])
Ps_gate = np.stack([Tvec[Tdays[Tdays.index(d) - 1]] for d in gate_days])  # template = dia anterior
Ft_gate = prevê_lstnet([int(np.where(ends == pd.Timestamp(d) + pd.Timedelta(hours=1, minutes=5))[0][0])
                        for d in gate_days], ft_state)
win = np.array([mae(Ft_gate[k:k+1], Tvec[d][None, :]) < mae(Ps_gate[k:k+1], Tvec[d][None, :])
                for k, d in enumerate(gate_days)])
print(f"dias em que o ft venceu no ajuste: {int(win.sum())}/{len(win)}")
print(pd.DataFrame({"dia": [str(d) for d in gate_days], "hind_mean7": Fgate[:, 0].round(3),
                    "ft_vence": win}).to_string())

## 8. gate-rule (1 parâmetro) + gate-lr (5 features)
Regra: `hind_mean7 > t` → desafiante; `t` maximiza acerto nos dias de ajuste. Logística nos 5 feats (alvo = `ft` vence). Parâmetros salvos em `modelos/gate.json` (pequeno, versionado).

In [ ]:
candidatos = np.unique(np.round(np.quantile(Fgate[:, 0], np.linspace(0, 1, 41)), 4))
best_t, best_acc = None, -1
for t in candidatos:
    acc = float((((Fgate[:, 0] > t).astype(int)) == win.astype(int)).mean())
    if acc > best_acc:
        best_t, best_acc = t, acc
print(f"gate-rule: t={best_t:.4f} acerto no ajuste={best_acc:.3f} ({int(best_acc*len(win))}/{len(win)})")

lr = LogisticRegression(max_iter=2000).fit(Fgate, win.astype(int))
print(f"gate-lr: acerto no ajuste={lr.score(Fgate, win):.3f} coef={lr.coef_.round(3).tolist()}")
json.dump({"t_rule": float(best_t), "acc_rule_train": best_acc,
           "coef_lr": lr.coef_.tolist(), "intercept_lr": lr.intercept_.tolist(),
           "acc_lr_train": float(lr.score(Fgate, win)), "feats": FEAT_NAMES,
           "gate_days": [str(d) for d in gate_days]},
          open(OUT / "modelos" / "gate.json", "w"))
print("gate salvo: modelos/gate.json")

## 9. Avaliação — `te` (sanity), holdout diário (primária) e 2594 origens (per-origin)
O gate decide por origem (origens do mesmo dia partilham os feats do dia). Braços: `rule_ft`, `lr_ft`, variante `rule_lgbm` (se houver pkl) e `oracle` (melhor ex-post entre os disponíveis — teto, não modelo).

In [ ]:
def feats_origens(idxs):
    ii = np.asarray(idxs)
    F = np.stack([feats_dia((e + pd.Timedelta(minutes=5)).date()) for e in ends[ii]])
    return F

def aplica_gate(F, P_ps, P_ft, P_lgbm=None, modo="rule"):
    if modo == "rule":
        use_ft = F[:, 0] > best_t
    else:
        use_ft = lr.predict(F).astype(bool)
    G = np.where(use_ft[:, None], P_ft, P_ps)
    out = {f"gate-{modo}_ft": G}
    if P_lgbm is not None:
        out[f"gate-{modo}_lgbm"] = np.where(use_ft[:, None], P_lgbm, P_ps)
    return out, use_ft

def avalia(idxs, tag):
    ii = np.asarray(idxs)
    Xb = X[ii]
    Ps = snaive(Xb)
    Ft = prevê_lstnet(ii, ft_state)
    Gb = prevê_lgbm(ii) if HAVE_LGBM else None
    F = feats_origens(ii)
    res = {"sazonal_naive_288": Ps, "lstnet_ft": Ft}
    if Gb is not None:
        res["lgbm_A"] = Gb
    for modo in ["rule", "lr"]:
        g, used = aplica_gate(F, Ps, Ft, Gb, modo)
        res.update(g)
        print(f"[{tag}/{modo}] origens com desafiante: {int(used.sum())}/{len(ii)}")
    cands = [Ps, Ft] + ([Gb] if Gb is not None else [])
    res["oracle"] = np.stack([cands[int(np.argmin([mae(c[k:k+1], Y[ii][k:k+1]) for c in cands]))][k]
                               for k in range(len(ii))])
    met = {m: metricas(Y[ii], p) for m, p in res.items()}
    print(f"=== {tag} ===")
    print(pd.DataFrame(met).T.round(4).to_string())
    return res, met, F

res_te, met_te, _ = avalia(te, "teste-434")
pd.DataFrame(met_te).T.round(4).to_csv(OUT / "metricas_baseline.csv")
res_da, met_da, F_da = avalia(daily_idx, "holdout-diário")
pd.DataFrame(met_da).T.round(4).to_csv(OUT / "metricas_holdout.csv")
res_ho, met_ho, F_ho = avalia(ho, "holdout-2594")
pd.DataFrame(met_ho).T.round(4).to_csv(OUT / "metricas_holdout_2594.csv")
print(f"\nRégua 00b (teste): 0.1525 | melhor gate: {min(met_te['gate-rule_ft']['MAE'], met_te['gate-lr_ft']['MAE']):.4f} | oracle: {met_te['oracle']['MAE']:.4f}")
print(f"Régua 00b (holdout): 0.1550 | melhor gate: {min(met_da['gate-rule_ft']['MAE'], met_da['gate-lr_ft']['MAE']):.4f} | oracle: {met_da['oracle']['MAE']:.4f}")
por_dia = pd.DataFrame(
    {"sazonal": [mae(Y[daily_idx][k:k+1], res_da["sazonal_naive_288"][k:k+1]) for k in range(len(daily_idx))],
     "lstnet_ft": [mae(Y[daily_idx][k:k+1], res_da["lstnet_ft"][k:k+1]) for k in range(len(daily_idx))],
     "gate-rule_ft": [mae(Y[daily_idx][k:k+1], res_da["gate-rule_ft"][k:k+1]) for k in range(len(daily_idx))],
     "gate-lr_ft": [mae(Y[daily_idx][k:k+1], res_da["gate-lr_ft"][k:k+1]) for k in range(len(daily_idx))],
     "oracle": [mae(Y[daily_idx][k:k+1], res_da["oracle"][k:k+1]) for k in range(len(daily_idx))],
     "hind_mean7": F_da[:, 0], "amp_dia": [float(Y[daily_idx][k].max() - Y[daily_idx][k].min()) for k in range(len(daily_idx))]}, 
    index=[str(ends[i].date()) for i in daily_idx])
por_dia.to_csv(OUT / "metricas_por_dia.csv")
print(por_dia.round(4).to_string())

In [ ]:
E = ends[te]
Ft_te = res_te["lstnet_ft"]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Ft_te[k], lw=1, alpha=0.6, label="lstnet_ft")
    ax.plot(tf, res_te["gate-rule_ft"][k], lw=1.2, alpha=0.9, label="gate-rule")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
pd.DataFrame(met_te).T["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

Yda = Y[daily_idx]
fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yda))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yda[k], "k-", lw=1.2, label="real")
    ax.plot(tf, res_da["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, res_da["lstnet_ft"][k], lw=1, alpha=0.6, label="lstnet_ft")
    ax.plot(tf, res_da["gate-rule_ft"][k], lw=1.2, alpha=0.9, label="gate-rule")
    ax.set_title(f"dia {ends[daily_idx[k]].date()} (MAE gate={por_dia['gate-rule_ft'].iloc[k]:.3f} vs saz={por_dia['sazonal'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")

# 07: calibração — hind_mean7 × erro real do sazonal (treino do gate + holdout)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(Fgate[:, 0], [mae(Tvec[d][None, :], Tvec[Tdays[Tdays.index(d)-1]][None, :]) for d in gate_days],
           s=60, alpha=0.7, label="dias de ajuste")
ax.scatter(por_dia["hind_mean7"].values, por_dia["sazonal"].values,
           s=60, marker="s", alpha=0.9, label="holdout (10 dias)")
ax.axvline(best_t, color="r", ls="--", label=f"t={best_t:.3f}")
ax.set_xlabel("hind_mean7 (instabilidade prevista)")
ax.set_ylabel("MAE real do sazonal no dia")
ax.set_title("Calibração do gate-rule")
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-gate-calibracao.png")

# 08: linha do tempo das chaves (quem o gate escolheu por dia) + acerto
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
rule_pick = ((por_dia["hind_mean7"] > best_t).astype(int))
axes[0].bar(range(len(por_dia)), por_dia["amp_dia"].values)
axes[0].set_title("Amplitude do dia real")
axes[0].set_xticks(range(len(por_dia)), por_dia.index.tolist(), rotation=30, fontsize=8)
axes[1].step(range(len(por_dia)), rule_pick.values, where="mid", lw=2, label="gate-rule: 0=sazonal 1=ft")
axes[1].scatter(range(len(por_dia)),
                (por_dia["lstnet_ft"] < por_dia["sazonal"]).astype(int).values,
                marker="x", s=60, label="oráculo (quem vencia)")
axes[1].set_title("Chaves por dia: gate vs oráculo")
axes[1].set_xticks(range(len(por_dia)), por_dia.index.tolist(), rotation=30, fontsize=8)
axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "08-chaves.png")
print("figs salvas")

## 10. Conclusões e próximos passos

- Réguas do 00b (0,1525 / 0,1550) impressas na §9; o gate decide por origem a partir de features 100% causais, ajustadas só em dias pré-`te`.
- Se algum gate vencer no holdout, vira a régua do OD; o `oracle` mostra quanto do gap é capturável por chaveamento perfeito.
- Se o gate empatar com o sazonal, a instabilidade dia-a-dia é imprevisível com histórico univariado — veredito "teto" definitivo para MAE em H=1d; próximos: quantis, transferência pós-gap, deploy.
- Artefatos em `resultados/07-gate-od/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `metricas_holdout_2594.csv`, `metricas_por_dia.csv`, `modelos/gate.json` e `figs/` (nenhum `.pkl` novo; `lgbm_A` reutilizado do 06 quando disponível).